In [1]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

START_YEAR = 2000
END_YEAR = 2020   # 2021 excluded: it's a partial year in this dataset
N_LAGS = 3

In [2]:
df = pd.read_csv("netflix_titles.csv")
df = df.drop_duplicates(subset="title").reset_index(drop=True)
df = df[(df["release_year"] >= START_YEAR) & (df["release_year"] <= END_YEAR)]

yearly = df.groupby("release_year").size().reset_index(name="title_count")
yearly = yearly.set_index("release_year").reindex(
    range(START_YEAR, END_YEAR + 1), fill_value=0
).reset_index().rename(columns={"index": "release_year"})

# Lag features -- lets a plain regression model "see" the recent trend,
# since it has no built-in concept of time order otherwise
for lag in range(1, N_LAGS + 1):
    yearly[f"lag_{lag}"] = yearly["title_count"].shift(lag)

yearly["rolling_mean_3"] = yearly["title_count"].shift(1).rolling(window=3).mean()
yearly["yoy_growth"] = yearly["title_count"].shift(1) - yearly["title_count"].shift(2)

yearly = yearly.dropna().reset_index(drop=True)
yearly

,release_year,title_count,lag_1,lag_2,lag_3,rolling_mean_3,yoy_growth
0,2003,59,51.0,45.0,37.0,44.333333,6.0
1,2004,64,59.0,51.0,45.0,51.666667,8.0
2,2005,80,64.0,59.0,51.0,58.000000,5.0
3,2006,96,80.0,64.0,59.0,67.666667,16.0
4,2007,88,96.0,80.0,64.0,80.000000,16.0
5,2008,135,88.0,96.0,80.0,88.000000,-8.0
6,2009,152,135.0,88.0,96.0,106.333333,47.0
7,2010,192,152.0,135.0,88.0,125.000000,17.0
8,2011,185,192.0,152.0,135.0,159.666667,40.0
9,2012,236,185.0,192.0,152.0,176.333333,-7.0


In [3]:
print(yearly[["release_year", "title_count"]].to_string(index=False))

growth = yearly["title_count"].pct_change().mean() * 100
print(f"\nAverage year-over-year growth rate: {growth:.1f}%")
peak_year = yearly.loc[yearly["title_count"].idxmax(), "release_year"]
peak_count = yearly["title_count"].max()
print(f"Peak release year: {int(peak_year)} ({int(peak_count)} titles)")

 release_year  title_count
         2003           59
         2004           64
         2005           80
         2006           96
         2007           88
         2008          135
         2009          152
         2010          192
         2011          185
         2012          236
         2013          286
         2014          352
         2015          555
         2016          901
         2017         1030
         2018         1144
         2019         1029
         2020          953

Average year-over-year growth rate: 19.6%
Peak release year: 2018 (1144 titles)


In [5]:
feature_cols = [c for c in yearly.columns if c not in ["release_year", "title_count"]]
X = yearly[feature_cols]
y = yearly["title_count"]
test_size = 4
X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]
print("Train years:", yearly["release_year"].iloc[:-test_size].tolist())
print("Test years: ", yearly["release_year"].iloc[-test_size:].tolist())

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=4)
rf.fit(X_train, y_train)

models = {"Linear Regression": lin_reg, "Random Forest": rf}

Train years: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
Test years:  [2017, 2018, 2019, 2020]


In [6]:
results = {}
for name, model in models.items():
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = mae
    print(f"\n--- {name} ---")
    print(f"MAE:  {mae:.1f} titles")
    print(f"RMSE: {rmse:.1f} titles")
    print(f"R²:   {r2:.3f}")
    print("Actual vs Predicted:", list(zip(y_test.astype(int), preds.round(1))))

best_name = min(results, key=results.get)
print(f"\nBest model (lowest MAE): {best_name}")


--- Linear Regression ---
MAE:  481.5 titles
RMSE: 493.4 titles
R²:   -51.348
Actual vs Predicted: [(1030, np.float64(1559.6)), (1144, np.float64(1678.6)), (1029, np.float64(1594.5)), (953, np.float64(1249.2))]

--- Random Forest ---
MAE:  315.9 titles
RMSE: 320.2 titles
R²:   -21.043
Actual vs Predicted: [(1030, np.float64(777.8)), (1144, np.float64(746.7)), (1029, np.float64(723.7)), (953, np.float64(644.1))]

Best model (lowest MAE): Random Forest


In [7]:
best_model = models[best_name]

def forecast_future(model, yearly, feature_cols, n_years=3):
    history = yearly["title_count"].tolist()
    predictions = []
    for i in range(n_years):
        lags = history[-N_LAGS:][::-1]
        rolling_mean_3 = np.mean(history[-3:])
        yoy_growth = history[-1] - history[-2]
        row = {f"lag_{n+1}": v for n, v in enumerate(lags)}
        row["rolling_mean_3"] = rolling_mean_3
        row["yoy_growth"] = yoy_growth
        X_next = pd.DataFrame([row])[feature_cols]
        pred = max(0, model.predict(X_next)[0])
        predictions.append(pred)
        history.append(pred)
    return predictions

future_preds = forecast_future(best_model, yearly, feature_cols, n_years=3)
last_year = int(yearly["release_year"].max())
for i, pred in enumerate(future_preds, start=1):
    print(f"Predicted title count for {last_year + i}: {pred:.0f}")

Predicted title count for 2021: 644
Predicted title count for 2022: 644
Predicted title count for 2023: 644
